In [1]:
import sys

import warnings
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm

from fadtk import (
    VGGishModel,
    calc_embd_statistics,
    calc_frechet_distance,
)

from configuration import (
    SAMPLE_RATE, N_FFT, HOP_LENGTH,
    LATENT_CHANNELS,
    VAE_CKPT_PATH, DIFFUSION_CKPT_PATH, LATENT_STATS_PATH,
    OUTPUT_DIR,
)
from dataset.create_chunks import iter_chunks
from models import VAE, UNetDenoiser, sample_latents, mel_tensor_to_audio

RESULTS_DIR = OUTPUT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MUSDB_ROOT     = Path('../musdb18/musdb18_mixtures')
CHUNK_DURATION = 10
N_GENERATE     = 200
N_REFERENCE    = 200
FAD_SR         = 16000

GEN_DIR  = OUTPUT_DIR / 'fad_generated'
REF_DIR  = OUTPUT_DIR / 'fad_reference'
REFA_DIR = OUTPUT_DIR / 'fad_ref_A'
REFB_DIR = OUTPUT_DIR / 'fad_ref_B'
for d in [GEN_DIR, REF_DIR, REFA_DIR, REFB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


/home/matej/Documents/DIPLRAD/music_gen_v2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:

vae = VAE().to(device)
vae.load_state_dict(torch.load(VAE_CKPT_PATH, map_location=device)['model_state'])
vae.eval()

diff_ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location=device)
unet      = UNetDenoiser().to(device)
unet.load_state_dict(diff_ckpt['model_state'])
unet.eval()

stats        = torch.load(LATENT_STATS_PATH, map_location='cpu')
latent_mean  = stats['latent_mean']
latent_std   = stats['latent_std']
latent_shape = stats['latent_shape']
mel_shape    = stats['mel_shape']

print('Models loaded.')
print(f'latent_shape={latent_shape}  mel_shape={mel_shape}')


Models loaded.
latent_shape=(16, 10, 27)  mel_shape=(80, 216)


In [3]:
ml = VGGishModel()
ml.load_model()
print(f'VGGishModel ready — sr={ml.sr}, features={ml.num_features}')


Using cache found in /home/matej/.cache/torch/hub/harritaylor_torchvggish_master


VGGishModel ready — sr=16000, features=128


In [4]:
def extract_embeddings(wav_paths: list, ml: VGGishModel) -> np.ndarray:

    all_embs = []
    skipped  = 0
    for p in tqdm(wav_paths, desc='Embedding', leave=False):
        try:
            audio = ml.load_wav(Path(p))
            emb   = ml.get_embedding(audio)
            all_embs.append(emb.astype(np.float32))
        except Exception as e:
            warnings.warn(f'Skipping {p}: {e}')
            skipped += 1
    if skipped:
        print(f'  Warning: skipped {skipped}/{len(wav_paths)} files')
    return np.vstack(all_embs) if all_embs else np.empty((0, ml.num_features), dtype=np.float32)


In [5]:
def save_resampled_wavs(src_list, out_dir: Path, prefix: str) -> list[str]:
    paths = []
    for i, src in enumerate(tqdm(src_list, desc=f'Writing {prefix}', leave=False)):
        audio, sr = librosa.load(src, sr=None, mono=True)
        if sr != FAD_SR:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=FAD_SR)
        out = out_dir / f'{prefix}_{i:04d}.wav'
        sf.write(out, audio.astype(np.float32), FAD_SR)
        paths.append(str(out))
    return paths


In [6]:
all_test = list(iter_chunks(MUSDB_ROOT, 'test', CHUNK_DURATION))
print(f'Total test chunks available: {len(all_test)}')

ref_files = all_test[:N_REFERENCE]
print(f'Using {len(ref_files)} clips for main reference')

reference_paths = save_resampled_wavs(ref_files, REF_DIR, 'ref')


if len(all_test) >= 2 * N_REFERENCE:
    refA_src = all_test[:N_REFERENCE]
    refB_src = all_test[N_REFERENCE: 2 * N_REFERENCE]
else:
    rng  = np.random.default_rng(42)
    idx  = rng.permutation(len(all_test))
    half = len(all_test) // 2
    refA_src = [all_test[i] for i in idx[:half]]
    refB_src = [all_test[i] for i in idx[half:]]
    print(f'  Self-FAD split: {len(refA_src)} vs {len(refB_src)} (random 50/50)')

refA_paths = save_resampled_wavs(refA_src, REFA_DIR, 'refA')
refB_paths = save_resampled_wavs(refB_src, REFB_DIR, 'refB')
print(f'Self-FAD sets ready: A={len(refA_paths)}, B={len(refB_paths)}')


Total test chunks available: 1223
Using 200 clips for main reference


Self-FAD sets ready: A=200, B=200


In [7]:
def mel_to_wav_resampled(mel_01_tensor, target_sr: int) -> np.ndarray:
    audio = mel_tensor_to_audio(mel_01_tensor)
    if target_sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=SAMPLE_RATE, target_sr=target_sr)
    return audio.astype(np.float32)

BATCH_GEN       = 4
generated_paths = []
n_done          = 0

with tqdm(total=N_GENERATE, desc='Generating') as pbar:
    while n_done < N_GENERATE:
        batch_size = min(BATCH_GEN, N_GENERATE - n_done)
        _, mel_batch = sample_latents(
            unet, vae,
            n_samples    = batch_size,
            latent_shape = latent_shape,
            latent_mean  = latent_mean,
            latent_std   = latent_std,
            mel_shape    = mel_shape,
        )
        for i in range(batch_size):
            audio    = mel_to_wav_resampled(mel_batch[i, 0], FAD_SR)
            out_path = GEN_DIR / f'gen_{n_done:04d}.wav'
            sf.write(out_path, audio, FAD_SR)
            generated_paths.append(str(out_path))
            n_done += 1
            pbar.update(1)

print(f'Generated {len(generated_paths)} files in {GEN_DIR}')


Generating: 100%|██████████| 200/200 [02:15<00:00,  1.48it/s]

Generated 200 files in ../musdb18/output/fad_generated


In [8]:
print('Embedding generated set...')
gen_embs  = extract_embeddings(generated_paths, ml)

print('Embedding reference set...')
ref_embs  = extract_embeddings(reference_paths, ml)

print('Embedding self-FAD set A...')
refA_embs = extract_embeddings(refA_paths, ml)

print('Embedding self-FAD set B...')
refB_embs = extract_embeddings(refB_paths, ml)

print(f'\nEmbedding shapes (rows >> 128 needed for stable covariance):')
for name, embs in [('gen', gen_embs), ('ref', ref_embs), ('refA', refA_embs), ('refB', refB_embs)]:
    status = 'ok' if embs.shape[0] >= 128 else 'increase N_GENERATE/N_REFERENCE'
    print(f'  {name:5s}: {embs.shape}  {status}')


Embedding generated set...


Embedding reference set...


Embedding self-FAD set A...


Embedding self-FAD set B...



Embedding shapes (rows >> 128 needed for stable covariance):
  gen  : (2000, 128)  ok
  ref  : (2000, 128)  ok
  refA : (2000, 128)  ok
  refB : (2000, 128)  ok


In [9]:
mu_gen,  cov_gen  = calc_embd_statistics(gen_embs)
mu_ref,  cov_ref  = calc_embd_statistics(ref_embs)
mu_refA, cov_refA = calc_embd_statistics(refA_embs)
mu_refB, cov_refB = calc_embd_statistics(refB_embs)

self_fad  = calc_frechet_distance(mu_refA, cov_refA, mu_refB, cov_refB)

model_fad = calc_frechet_distance(mu_gen, cov_gen, mu_ref, cov_ref)

print(f'Self-FAD  (MUSDB→MUSDB, lower bound) : {self_fad:.4f}')
print(f'Model FAD (generated→MUSDB)          : {model_fad:.4f}')
print(f'Ratio model / self                   : {model_fad / (self_fad + 1e-9):.1f}x')



Self-FAD  (MUSDB→MUSDB, lower bound) : 1.9123
Model FAD (generated→MUSDB)          : 15.7171
Ratio model / self                   : 8.2x


In [10]:
# ── Save result ─────────────────────────────────────────────────────────────────
result_text = (
    f'FAD Evaluation\n'
    f'==============\n'
    f'Library              : fadtk (VGGishModel)\n'
    f'Generated samples    : {N_GENERATE}\n'
    f'Reference samples    : {N_REFERENCE}\n'
    f'Embedding SR         : {FAD_SR} Hz (resampled from {SAMPLE_RATE} Hz)\n'
    f'Embedding dim        : {ml.num_features}\n'
    f'\n'
    f'Embedding row counts :\n'
    f'  generated : {gen_embs.shape[0]}\n'
    f'  reference : {ref_embs.shape[0]}\n'
    f'  refA      : {refA_embs.shape[0]}\n'
    f'  refB      : {refB_embs.shape[0]}\n'
    f'\n'
    f'Self-FAD  (MUSDB→MUSDB, lower bound) : {self_fad:.4f}\n'
    f'Model FAD (generated→MUSDB)          : {model_fad:.4f}\n'
    f'Ratio model/self                     : {model_fad / (self_fad + 1e-9):.2f}x\n'
)

out_path = RESULTS_DIR / 'fad_score.txt'
out_path.write_text(result_text)
print(result_text)
print('Saved:', out_path)


FAD Evaluation
Library              : fadtk (VGGishModel)
Generated samples    : 200
Reference samples    : 200
Embedding SR         : 16000 Hz (resampled from 5512 Hz)
Embedding dim        : 128

Embedding row counts :
  generated : 2000
  reference : 2000
  refA      : 2000
  refB      : 2000

Self-FAD  (MUSDB→MUSDB, lower bound) : 1.9123
Model FAD (generated→MUSDB)          : 15.7171
Ratio model/self                     : 8.22x

Saved: ../musdb18/output/results/fad_score.txt
